# Model 7: Girls Reintegration Readiness Pipeline

## 1. Problem Framing
- **Business question:** Which residents appear most ready for successful reintegration so case managers can prioritize action plans?
- **Stakeholders:** Admin leadership, social workers, and safehouse case conference teams.
- **Predictive objective:** Estimate each resident's readiness likelihood and rank top candidates for weekly worklists.
- **Explanatory companion:** Identify the strongest associated factors (risk profile, process progress, education, health, incidents) to support intervention planning.
- **Decision implication:** Higher readiness residents can move toward reintegration steps while low/medium groups receive targeted support.

## 2. Data Acquisition, Preparation & Exploration
- Data sources are loaded from `residents`, `safehouses`, `process_recordings`, `home_visitations`, `education_records`, `health_wellbeing_records`, `intervention_plans`, and `incident_reports`.
- The production pipeline aggregates resident-level features (counts and means) so training and inference use the same schema.
- Numeric values embedded in text fields (`present_age`, `length_of_stay`) are parsed into numeric features.
- Labeling logic maps `reintegration_status` text into binary readiness labels for supervised training.
- EDA checks should include missingness, class balance, and feature distributions before each major retraining run.

## 3. Modeling & Feature Selection
- **Predictive model:** Random forest classifier for non-linear interactions and robust ranking quality.
- **Explanatory companion model:** Logistic regression with the same feature space for directional interpretability.
- Feature set intentionally combines case, service, education, wellbeing, and safety domains to reflect holistic readiness.
- Candidate features are reviewed and trimmed to avoid leakage and unstable proxy variables.

## 4. Evaluation & Interpretation
- Validation uses a held-out test split with ROC-AUC, average precision, F1, and positive-rate diagnostics.
- Business interpretation focuses on worklist reliability: the top-ranked residents should be the most likely to succeed when reviewed by staff.
- False positives and false negatives are both operationally meaningful and should be monitored in case conferences.

## 5. Causal and Relationship Analysis
- This pipeline is primarily **predictive** and does **not** claim causal proof.
- Feature importance and coefficient directions indicate associations, not guaranteed interventions.
- Plausible relationships (higher progress indicators and lower unresolved incidents correlating with readiness) are used as hypothesis support for practitioner judgment.
- Staff decisions should combine model output with social-worker context and safeguarding constraints.

## 6. Deployment Notes
- **Production does not execute this notebook.** The live app runs `backend/ML/girls_reintegration_train.py` via the ASP.NET API (same pattern as other ML pipelines). This notebook documents the pipeline for class submission and optional local experimentation only.
- Backend endpoint `POST /api/ml/girls-reintegration/train` triggers training from live database data.
- Backend endpoint `GET /api/ml/girls-reintegration/insights` serves the latest readiness distribution, worklist, metrics, and key features.
- Frontend page `GirlsReintegrationInsightsPage` in the admin portal displays KPIs, top resident worklist, feature importances, and retrain controls.
- Artifacts are stored under `backend/ML/artifacts/girls_reintegration_mlr_latest.json` for stable retrieval.
- If training fails, the API returns JSON with `error`, `code`, and `hint` (and the UI shows the message). Common codes: `insufficient_labeled_rows`, `single_class_labels`, `python_training_failed`.


In [ ]:
# Optional notebook validation helper:
# Confirm endpoint contract after local backend startup.
import requests

BASE_URL = "http://localhost:5083"
print("Train endpoint:", f"{BASE_URL}/api/ml/girls-reintegration/train")
print("Insights endpoint:", f"{BASE_URL}/api/ml/girls-reintegration/insights")
